In [1]:
from functools import partial
from notebooks._utils import report_series_ensemble_accuracy_by_nparas
from notebooks._utils import calculate_parallel_ensemble_accuracy
from notebooks._utils import calculate_baseline_accuracy

ds_name = "myriadlama"
dataset_root = "/home/xzhao/workspace/GYB_self-ensemble/datasets"

In [2]:
def get_layers(model: str):
    if model.startswith("llama3.2_1b"):
        layers = 12
    elif model.startswith("llama3.2_3b"):
        layers = 21
    elif model.startswith("llama3.1_8b"):
        layers = 24
    elif model.startswith("qwen2.5_3b"):
        layers = 27
    elif model.startswith("qwen2.5_7b"):
        layers = 21
    elif model.startswith("qwen2.5_14b"):
        layers = 36
    else:
        raise NotImplementedError(f"Layers not defined for model {model}")
    return layers


In [4]:
num_fewshots = 5
for model_name in ["llama3.2_1b", "llama3.2_1b_it", "llama3.2_3b", "llama3.2_3b_it", "llama3.1_8b", "llama3.1_8b_it"]:
    print(f"\n=================== Model: {model_name} ===================")
    dump_file_prefix = f"{dataset_root}/{ds_name}/{model_name}/myriadlama."
    report_accuracy = partial(
        report_series_ensemble_accuracy_by_nparas, 
        dump_file_prefix=dump_file_prefix,
        single_para_qapair=True,
        explicit_prompts=False,
        repeat_paras=False, 
        num_fewshots=num_fewshots, use_generated=True)
    
    print("---- Calculating baseline ----")
    calculate_baseline_accuracy(dataset_root, model_name, num_fewshots)

    report_acc_base_setting = partial(
        calculate_parallel_ensemble_accuracy, 
        dump_file_prefix=dump_file_prefix, repeat_paras=False,
        num_paraphrases=5, num_fewshots=num_fewshots, use_generation=True)

    print("\n---- Logits-based Ensemble (Average) ----")
    report_acc_base_setting(logits_ensemble_method="avg")
    
    print("\n---- Logits-based Ensemble (Maximum) ----")
    report_acc_base_setting(logits_ensemble_method="max")

    print("\n---- Logits-based Ensemble + Averaged Layer Output ----")
    multip_layavg_report = partial(report_acc_base_setting, logits_ensemble_method="avg", ensemble_method="layer_output_avg", multilayer=True)
    multip_layavg_report(token_mode="last", ensemble_alpha=1, ensemble_layer=get_layers(model_name))

    print("\n---- Logits-based Ensemble + Averaged FFN Activation ----")
    multip_layavg_report = partial(report_acc_base_setting, logits_ensemble_method="avg", ensemble_method="ffn_activation_avg", multilayer=True)
    multip_layavg_report(token_mode="last", ensemble_alpha=1, ensemble_layer=get_layers(model_name))

    print("\n---- Logits-based Ensemble + Maximum FFN Activation ----")
    multip_layavg_report = partial(report_acc_base_setting, logits_ensemble_method="avg", ensemble_method="ffn_activation_max", multilayer=True)
    multip_layavg_report(token_mode="last", ensemble_alpha=1, ensemble_layer=get_layers(model_name))



=================== Model: llama3.2_1b ===================
---- Calculating baseline ----
Acc: 0.3942 ==> 🏷️ baseline

---- Logits-based Ensemble (Average) ----
Acc: 0.4685 ==> 🏷️ 5paras 5shots  None layerNone  alpha1.0 token-all

---- Logits-based Ensemble (Maximum) ----
Acc: 0.4521 ==> 🏷️ 5paras 5shots  None layerNone  alpha1.0 token-all

---- Logits-based Ensemble + Averaged Layer Output ----
Acc: 0.4802 ==> 🏷️ 5paras 5shots  layer_output_avg layer12 Multilayer alpha1 token-last

---- Logits-based Ensemble + Averaged FFN Activation ----
Acc: 0.4742 ==> 🏷️ 5paras 5shots  ffn_activation_avg layer12 Multilayer alpha1 token-last

---- Logits-based Ensemble + Maximum FFN Activation ----
Acc: 0.3895 ==> 🏷️ 5paras 5shots  ffn_activation_max layer12 Multilayer alpha1 token-last

=================== Model: llama3.2_1b_it ===================
---- Calculating baseline ----
Acc: 0.3527 ==> 🏷️ baseline

---- Logits-based Ensemble (Average) ----
Acc: 0.4182 ==> 🏷️ 5paras 5shots  None layerNone  